In [36]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# dx = 1 km; Np = 1M; Nt = 5 min
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_5min.nc') #***
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_5min_1e6.nc') #***
res='1km';t_res='5min'
Np_str='1e6'

# # dx = 1km; Np = 50M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
# res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 100M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_100M.nc') #***
# res='1km'; t_res='1min'; Np_str='100e6'

# dx = 250 m
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***


job_array=False;index_adjust=0
ocean_fraction=0.25

In [37]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

times=data['time'].values/(1e9 * 60); times=times.astype(float);
minutes=1/times[1] #1 / minutes per timestep = timesteps per minute
kms=np.argmax(data['xh'].values-data['xh'][0].values >= 1)

In [38]:
def check_memory():
    import sys
    ipython_vars = ["In", "Out", "exit", "quit", "get_ipython", "ipython_vars"]
    print("Top 10 objects with highest memory usage")
    # Get a sorted list of the objects and their sizes
    mem = {
        key: round(value/1e6,2)
        for key, value in sorted(
            [
                (x, sys.getsizeof(globals().get(x)))
                for x in globals()
                if not x.startswith("_") and x not in sys.modules and x not in ipython_vars
            ],
            key=lambda x: x[1],
            reverse=True)[:10]
    }
    print({key:f"{value} MB" for key,value in mem.items()})
    print(f"\n{round(sum(mem.values()),2)/1000} GB in use overall")

In [20]:
#JOB ARRAY SETUP
job_array=True
if job_array==True:

    num_jobs=300 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
    total_elements=len(parcel['xh']) #total num of variables

    if num_jobs >= total_elements:
        raise ValueError("Number of jobs cannot be greater than or equal to total elements.")
    
    job_range = total_elements // num_jobs  # Base size for each chunk
    remaining = total_elements % num_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id, num_jobs):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_jobs - 1: 
            end_job = total_elements #- 1
        return start_job, end_job
    # def job_testing():
    #     #TESTING
    #     start=[];end=[]
    #     for job_id in range(1,num_jobs+1):
    #         start_job, end_job = get_job_range(job_id)
    #         print(start_job,end_job)
    #         start.append(start_job)
    #         end.append(end_job)
    #     print(np.all(start!=end))
    #     print(len(np.unique(start))==len(start))
    #     print(len(np.unique(end))==len(end))
    # job_testing()
    
    job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    if job_id==0: job_id=2
    start_job, end_job = get_job_range(job_id, num_jobs)
    index_adjust=start_job
    print(f'start_job = {start_job}, end_job = {end_job}')

start_job = 3334, end_job = 6668


In [8]:
#Indexing Array with JobArray
parcel=parcel.isel(xh=slice(start_job,end_job))
#(for 150_000_000 parcels use 500-1000 jobs)

In [40]:
# Reading Back Data Later
##############
def make_data_dict(var_names,read_type):
    if read_type=='h5py':
        with h5py.File(in_file, 'r') as f:
            data_dict = {var_name: f[var_name][:,start_job:end_job] for var_name in var_names}
            
    elif read_type=='xarray':
        in_data = xr.open_dataset(
            in_file,
            engine='h5netcdf',
            phony_dims='sort',
            chunks={'phony_dim_0': 100, 'phony_dim_1': 1_000_000} 
        )
        data_dict = {k: in_data[k][:,start_job:end_job].compute().data for k in var_names}
    return data_dict

# read_type='xarray'
read_type='h5py'

In [41]:
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'lagrangian_binary_array_{res}_{t_res}_{Np_str}.h5'

var_names = ['A_g', 'A_c', 'W', 'QCQI', 'Z', 'Y', 'X', 'z']
data_dict = make_data_dict(var_names,read_type)
A_g, A_c, W, QCQI, Z, Y, X, parcel_z = (data_dict[k] for k in var_names)

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)
check_memory()

Top 10 objects with highest memory usage
{'A_g': '1064.0 MB', 'A_c': '1064.0 MB', 'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'W': '532.0 MB', 'QCQI': '532.0 MB', 'parcel_z': '532.0 MB', 'QV': '1.77 MB', 'TH': '1.77 MB'}

6.91954 GB in use overall


In [42]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'VARS_binary_array_{res}_{t_res}_{Np_str}.h5'

var_names = ['QV','TH','TH_E','BUOYANCY','HMC']
data_dict = make_data_dict(var_names,read_type)
QV, TH, TH_E, BUOYANCY, HMC = (data_dict[k] for k in var_names)
check_memory()

Top 10 objects with highest memory usage
{'A_g': '1064.0 MB', 'A_c': '1064.0 MB', 'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'W': '532.0 MB', 'QCQI': '532.0 MB', 'parcel_z': '532.0 MB', 'QV': '532.0 MB', 'TH': '532.0 MB'}

7.98 GB in use overall


In [10]:
################################################################################

In [ ]:
########################
#READING BACK IN
def LoadFinalData(in_file):
    dict = {}
    with h5py.File(in_file, 'r') as f:
        for key in f.keys():
            dict[key] = f[key][:]
    return dict

def LoadAllCloudBase():
    dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/'
    in_file = dir2 + f"all_cloudbase_{res}_{t_res}_{Np_str}.pkl"
    with open(in_file, 'rb') as f:
        all_cloudbase = pickle.load(f)
    return(all_cloudbase)
min_all_cloudbase=np.nanmin(LoadAllCloudBase())
print(f"Minimum Cloudbase is: {min_all_cloudbase}\n")

dir2 = dir + f'Project_Algorithms/Tracking_Algorithms/'
in_file=dir2+f"parcel_tracking_SUBSET_{res}_{t_res}_{Np_str}"
final_dict=LoadFinalData(in_file)


#DYNAMICALLY CREATING VARIABLES
for key, value in final_dict.items():
    globals()[key] = value

# #DYNAMICALLY PRINTING VARIABLE SIZES
# for key in final_dict:
#     print(f"{key} has {final_dict[key].shape[0]} parcels")

#PRINTING VARIABLE SIZES (ONE BY ONE)
print(f'ALL: {len(ALL_out_arr)} CL parcels and {len(ALL_save_arr)} nonCL parcels')
print(f'SHALLOW: {len(SHALLOW_out_arr)} CL parcels and {len(SHALLOW_save_arr)} nonCL parcels')
print(f'DEEP: {len(DEEP_out_arr)} CL parcels and {len(DEEP_save_arr)} nonCL parcels')
print('\n')
print(f'ALL: {len(ALL_SBZ_out_arr)} SBZ parcels and {len(ALL_nonSBZ_out_arr)} nonSBZ parcels')
print(f'SHALLOW: {len(SHALLOW_SBZ_out_arr)} SBZ parcels and {len(SHALLOW_nonSBZ_out_arr)} nonSBZ parcels')
print(f'DEEP: {len(DEEP_SBZ_out_arr)} SBZ parcels and {len(DEEP_nonSBZ_out_arr)} nonSBZ parcels')
print('\n')
print(f'ALL: {len(ALL_ColdPool_out_arr)} ColdPool parcels')
print(f'SHALLOW: {len(SHALLOW_ColdPool_out_arr)} ColdPool parcels')
print(f'DEEP: {len(DEEP_ColdPool_out_arr)} ColdPool parcels')

#APPLYING JOB ARRAY
if "job_array" in globals():
    print('APPLYING JOB ARRAY')
    def job_filter(arr):
        return arr[(arr[:,0]>=start_job)&(arr[:,0]<end_job)]
    for name in [
        'ALL_out_arr', 'ALL_save_arr',
        'SHALLOW_out_arr', 'SHALLOW_save_arr',
        'DEEP_out_arr', 'DEEP_save_arr',
        'ALL_SBZ_out_arr', 'ALL_nonSBZ_out_arr',
        'SHALLOW_SBZ_out_arr', 'SHALLOW_nonSBZ_out_arr',
        'DEEP_SBZ_out_arr', 'DEEP_nonSBZ_out_arr',
        'ALL_ColdPool_out_arr', 'SHALLOW_ColdPool_out_arr', 'DEEP_ColdPool_out_arr'
    ]:
        globals()[name] = job_filter(globals()[name])

In [16]:
#ALL/DEEP/SHALLOW CL vs non-CL Tracked Parcel Plots
################################################################################

In [29]:
def averaged_profiles(profile): 
    out_var=profile[ (profile[:, 1] != 0)]; #gets rid of rows that have no data
    out_var=np.array([out_var[:, 0] / out_var[:, 1], out_var[:, 2]]).T #divides the data column by the counter column
    return out_var
def averaged_profile_count(profile): 
    out_var=profile[ (profile[:, 1] != 0)]; #gets rid of rows that have no data
    counts=out_var[:, 1]
    zlevels=out_var[:, 2]
    return counts,zlevels

In [30]:
def CL_tracked_profile(var_data,type):
    if type=='all':
        out_arr=ALL_out_arr.copy()
        # after_array=ALL_out_after_array
    elif type=='shallow':
        out_arr=SHALLOW_out_arr.copy()
        # after_array=SHALLOW_out_after_array
    elif type=='deep':
        out_arr=DEEP_out_arr.copy()
        # after_array=DEEP_out_after_array

    
    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;
    
    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust]
        ys=Y[ts,p-index_adjust]
        xs=X[ts,p-index_adjust]
        
        vars=var_data[ts,p-index_adjust]
        np.add.at(profile_array[:, 0], zs, vars)
        np.add.at(profile_array[:, 1], zs, 1)
    return profile_array

print('ALL')
ALL_profile_array_w=CL_tracked_profile(W,type='all')
ALL_profile_array_qv=CL_tracked_profile(QV,type='all')
ALL_profile_array_qcqi=CL_tracked_profile(QCQI,type='all')
ALL_profile_array_th=CL_tracked_profile(TH,type='all')
ALL_profile_array_th_e=CL_tracked_profile(TH_E,type='all')
ALL_profile_array_buoyancy=CL_tracked_profile(BUOYANCY,type='all')
ALL_profile_array_HMC=CL_tracked_profile(HMC,type='all')
print('SHALLOW')
SHALLOW_profile_array_w=CL_tracked_profile(W,type='shallow')
SHALLOW_profile_array_qv=CL_tracked_profile(QV,type='shallow')
SHALLOW_profile_array_qcqi=CL_tracked_profile(QCQI,type='shallow')
SHALLOW_profile_array_th=CL_tracked_profile(TH,type='shallow')
SHALLOW_profile_array_th_e=CL_tracked_profile(TH_E,type='shallow')
SHALLOW_profile_array_buoyancy=CL_tracked_profile(BUOYANCY,type='shallow')
SHALLOW_profile_array_HMC=CL_tracked_profile(HMC,type='shallow')
print('DEEP')
DEEP_profile_array_w=CL_tracked_profile(W,type='deep')
DEEP_profile_array_qv=CL_tracked_profile(QV,type='deep')
DEEP_profile_array_qcqi=CL_tracked_profile(QCQI,type='deep')
DEEP_profile_array_th=CL_tracked_profile(TH,type='deep')
DEEP_profile_array_th_e=CL_tracked_profile(TH_E,type='deep')
DEEP_profile_array_buoyancy=CL_tracked_profile(BUOYANCY,type='deep')
DEEP_profile_array_HMC=CL_tracked_profile(HMC,type='deep')

ALL
SHALLOW
DEEP


In [31]:
POST_JOB_ARRAY=False
# POST_JOB_ARRAY=True

if POST_JOB_ARRAY==True:
    
    types = ["ALL", "SHALLOW", "DEEP"]
    variables = ["w", "qv", "qcqi", "th", "th_e", "buoyancy", "HMC"]
    
    vars_list = []
    SE_list = []
    
    for t in types:
        for var in variables:
            vars_list.append(f"{t}_profile_array_{var}")
            vars_list.append(f"nonCL_{t}_profile_array_{var}")
    for t in types:
        for var in variables:
            SE_list.append(f"{t}_SE_{var}")
            SE_list.append(f"nonCL_{t}_SE_{var}")
    
    # Define directory and output file path
    dir2 = dir + 'Project_Algorithms/Tracked_Profiles/job_out2/'
    output_file = dir2 + f"CL_nonCL_tracked_profiles_{res}_{Np_str}.h5"
    
    # Open the HDF5 file and read the stored datasets into dynamically named variables
    with h5py.File(output_file, 'r') as f:
        for var in vars_list:# + SE_list:
            globals()[var] = f[f'{var}'][:]

def averaged_profile_SE(profile): 

    ######################################## (MOVED TO SE_averaged_profiles FUNCTION)
    where_undefined=np.where(profile[:,1]==1)
    mask=np.where(profile[:,1]!=0)
    #DIVIDE BY N = n-1
    with np.errstate(divide='ignore', invalid='ignore'):
        profile[mask, 0] /= (profile[mask, 1]-1) 
    #TAKE THE SQUARE ROOT
    profile[mask,0]=np.sqrt(profile[mask,0]) 
    #DIVIDE BY SQUARE ROOT OF n ==> STANDARD ERROR
    profile[mask, 0] /= np.sqrt(profile[mask, 1]) 
    profile[where_undefined,0]=0
    ########################################
    out_var=profile[ (profile[:, 1] != 0)]; #gets rid of rows that have no data
    SE=out_var[:, 0]
    zlevels=out_var[:, 2]
    return SE,zlevels
    
def CL_tracked_profile_SE(profile_data,var_data,type):  
    global test
    if type=='all':
        out_arr=ALL_out_arr.copy()
        # after_array=ALL_out_after_array
    elif type=='shallow':
        out_arr=SHALLOW_out_arr.copy()
        # after_array=SHALLOW_out_after_array
    elif type=='deep':
        out_arr=DEEP_out_arr.copy()
        # after_array=DEEP_out_after_array

    
    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;

    # test=[] #TESTING
    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust]
        ys=Y[ts,p-index_adjust]
        xs=X[ts,p-index_adjust]
        
        vars=var_data[ts,p-index_adjust]
        mean_mu=profile_data[zs,0]/profile_data[zs,1]

        # for count,z in enumerate(zs): #TESTING
        #     if z==28:
        #         test.append(np.array(vars)[count])
        np.add.at(profile_array[:, 0], zs,  (vars-mean_mu)**2) #SUMMING UP THE SQUARES
        np.add.at(profile_array[:, 1], zs,  1) #SUMMING UP THE SQUARES
    return profile_array


ALL_SE_w=CL_tracked_profile_SE(ALL_profile_array_w,W,type='all')
SHALLOW_SE_w=CL_tracked_profile_SE(SHALLOW_profile_array_w,W,type='shallow')
DEEP_SE_w=CL_tracked_profile_SE(DEEP_profile_array_w,W,type='deep')

ALL_SE_qv=CL_tracked_profile_SE(ALL_profile_array_qv,QV,type='all')
SHALLOW_SE_qv=CL_tracked_profile_SE(SHALLOW_profile_array_qv,QV,type='shallow')
DEEP_SE_qv=CL_tracked_profile_SE(DEEP_profile_array_qv,QV,type='deep')

ALL_SE_qcqi=CL_tracked_profile_SE(ALL_profile_array_qcqi,QCQI,type='all')
SHALLOW_SE_qcqi=CL_tracked_profile_SE(SHALLOW_profile_array_qcqi,QCQI,type='shallow')
DEEP_SE_qcqi=CL_tracked_profile_SE(DEEP_profile_array_qcqi,QCQI,type='deep')

ALL_SE_th=CL_tracked_profile_SE(ALL_profile_array_th,TH,type='all')
SHALLOW_SE_th=CL_tracked_profile_SE(SHALLOW_profile_array_th,TH,type='shallow')
DEEP_SE_th=CL_tracked_profile_SE(DEEP_profile_array_th,TH,type='deep')

ALL_SE_th_e=CL_tracked_profile_SE(ALL_profile_array_th_e,TH_E,type='all')
SHALLOW_SE_th_e=CL_tracked_profile_SE(SHALLOW_profile_array_th_e,TH_E,type='shallow')
DEEP_SE_th_e=CL_tracked_profile_SE(DEEP_profile_array_th_e,TH_E,type='deep')

ALL_SE_buoyancy=CL_tracked_profile_SE(ALL_profile_array_buoyancy,BUOYANCY,type='all')
SHALLOW_SE_buoyancy=CL_tracked_profile_SE(SHALLOW_profile_array_buoyancy,BUOYANCY,type='shallow')
DEEP_SE_buoyancy=CL_tracked_profile_SE(DEEP_profile_array_buoyancy,BUOYANCY,type='deep')

ALL_SE_HMC=CL_tracked_profile_SE(ALL_profile_array_HMC,HMC,type='all')
SHALLOW_SE_HMC=CL_tracked_profile_SE(SHALLOW_profile_array_HMC,HMC,type='shallow')
DEEP_SE_HMC=CL_tracked_profile_SE(DEEP_profile_array_HMC,HMC,type='deep')

In [32]:
#MAKING PROFILE ARRAY

def nonCL_tracked_profile(var_data,type):
    if type=='all':
        out_arr=ALL_save_arr.copy()
        # after_array=ALL_save_after_array
    elif type=='shallow':
        out_arr=SHALLOW_save_arr.copy()
        # after_array=SHALLOW_save_after_array
    elif type=='deep':
        out_arr=DEEP_save_arr.copy()
        # after_array=DEEP_save_after_array
    
    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;
    
    for row in range(out_arr.shape[0]):
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        after=out_arr[row,3]
        p=out_arr[row,0]

        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust]
        ys=Y[ts,p-index_adjust]
        xs=X[ts,p-index_adjust]
        
        vars=var_data[ts,p-index_adjust]
        np.add.at(profile_array[:, 0], zs, vars)
        np.add.at(profile_array[:, 1], zs, 1)
    return profile_array

print('ALL')
nonCL_ALL_profile_array_w=nonCL_tracked_profile(W,type='all')
nonCL_ALL_profile_array_qv=nonCL_tracked_profile(QV,type='all')
nonCL_ALL_profile_array_qcqi=nonCL_tracked_profile(QCQI,type='all')
nonCL_ALL_profile_array_th=nonCL_tracked_profile(TH,type='all')
nonCL_ALL_profile_array_th_e=nonCL_tracked_profile(TH_E,type='all')
nonCL_ALL_profile_array_buoyancy=nonCL_tracked_profile(BUOYANCY,type='all')
nonCL_ALL_profile_array_HMC=nonCL_tracked_profile(HMC,type='all')
print('SHALLOW')
nonCL_SHALLOW_profile_array_w=nonCL_tracked_profile(W,type='shallow')
nonCL_SHALLOW_profile_array_qv=nonCL_tracked_profile(QV,type='shallow')
nonCL_SHALLOW_profile_array_qcqi=nonCL_tracked_profile(QCQI,type='shallow')
nonCL_SHALLOW_profile_array_th=nonCL_tracked_profile(TH,type='shallow')
nonCL_SHALLOW_profile_array_th_e=nonCL_tracked_profile(TH_E,type='shallow')
nonCL_SHALLOW_profile_array_buoyancy=nonCL_tracked_profile(BUOYANCY,type='shallow')
nonCL_SHALLOW_profile_array_HMC=nonCL_tracked_profile(HMC,type='shallow')
print('DEEP')
nonCL_DEEP_profile_array_w=nonCL_tracked_profile(W,type='deep')
nonCL_DEEP_profile_array_qv=nonCL_tracked_profile(QV,type='deep')
nonCL_DEEP_profile_array_qcqi=nonCL_tracked_profile(QCQI,type='deep')
nonCL_DEEP_profile_array_th=nonCL_tracked_profile(TH,type='deep')
nonCL_DEEP_profile_array_th_e=nonCL_tracked_profile(TH_E,type='deep')
nonCL_DEEP_profile_array_buoyancy=nonCL_tracked_profile(BUOYANCY,type='deep')
nonCL_DEEP_profile_array_HMC=nonCL_tracked_profile(HMC,type='deep')

ALL
SHALLOW
DEEP


In [33]:
#TESTING
def nonCL_tracked_profile_SE(profile_data,var_data,type):    
    if type=='all':
        out_arr=ALL_save_arr.copy()
        # after_array=ALL_save_after_array
    elif type=='shallow':
        out_arr=SHALLOW_save_arr.copy()
        # after_array=SHALLOW_save_after_array
    elif type=='deep':
        out_arr=DEEP_save_arr.copy()
        # after_array=DEEP_save_after_array

    
    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;

    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust]
        ys=Y[ts,p-index_adjust]
        xs=X[ts,p-index_adjust]
        
        vars=var_data[ts,p-index_adjust]
        mean_mu=profile_data[zs,0]/profile_data[zs,1]
        np.add.at(profile_array[:, 0], zs,  (vars-mean_mu)**2) #SUMMING UP THE SQUARES
        np.add.at(profile_array[:, 1], zs,  1) #SUMMING UP THE SQUARES

    return profile_array


nonCL_ALL_SE_w=nonCL_tracked_profile_SE(nonCL_ALL_profile_array_w,W,type='all')
nonCL_SHALLOW_SE_w=nonCL_tracked_profile_SE(nonCL_SHALLOW_profile_array_w,W,type='shallow')
nonCL_DEEP_SE_w=nonCL_tracked_profile_SE(nonCL_DEEP_profile_array_w,W,type='deep')

nonCL_ALL_SE_qv=nonCL_tracked_profile_SE(nonCL_ALL_profile_array_qv,QV,type='all')
nonCL_SHALLOW_SE_qv=nonCL_tracked_profile_SE(nonCL_SHALLOW_profile_array_qv,QV,type='shallow')
nonCL_DEEP_SE_qv=nonCL_tracked_profile_SE(nonCL_DEEP_profile_array_qv,QV,type='deep')

nonCL_ALL_SE_qcqi=nonCL_tracked_profile_SE(nonCL_ALL_profile_array_qcqi,QCQI,type='all')
nonCL_SHALLOW_SE_qcqi=nonCL_tracked_profile_SE(nonCL_SHALLOW_profile_array_qcqi,QCQI,type='shallow')
nonCL_DEEP_SE_qcqi=nonCL_tracked_profile_SE(nonCL_DEEP_profile_array_qcqi,QCQI,type='deep')

nonCL_ALL_SE_th=nonCL_tracked_profile_SE(nonCL_ALL_profile_array_th,TH,type='all')
nonCL_SHALLOW_SE_th=nonCL_tracked_profile_SE(nonCL_SHALLOW_profile_array_th,TH,type='shallow')
nonCL_DEEP_SE_th=nonCL_tracked_profile_SE(nonCL_DEEP_profile_array_th,TH,type='deep')

nonCL_ALL_SE_th_e=nonCL_tracked_profile_SE(nonCL_ALL_profile_array_th_e,TH_E,type='all')
nonCL_SHALLOW_SE_th_e=nonCL_tracked_profile_SE(nonCL_SHALLOW_profile_array_th_e,TH_E,type='shallow')
nonCL_DEEP_SE_th_e=nonCL_tracked_profile_SE(nonCL_DEEP_profile_array_th_e,TH_E,type='deep')

nonCL_ALL_SE_buoyancy=nonCL_tracked_profile_SE(nonCL_ALL_profile_array_buoyancy,BUOYANCY,type='all')
nonCL_SHALLOW_SE_buoyancy=nonCL_tracked_profile_SE(nonCL_SHALLOW_profile_array_buoyancy,BUOYANCY,type='shallow')
nonCL_DEEP_SE_buoyancy=nonCL_tracked_profile_SE(nonCL_DEEP_profile_array_buoyancy,BUOYANCY,type='deep')

nonCL_ALL_SE_HMC=nonCL_tracked_profile_SE(nonCL_ALL_profile_array_HMC,HMC,type='all')
nonCL_SHALLOW_SE_HMC=nonCL_tracked_profile_SE(nonCL_SHALLOW_profile_array_HMC,HMC,type='shallow')
nonCL_DEEP_SE_HMC=nonCL_tracked_profile_SE(nonCL_DEEP_profile_array_HMC,HMC,type='deep')

In [34]:
#SAVING

# Define categories and variables
types = ["ALL", "SHALLOW", "DEEP"]
variables = ["w", "qv", "qcqi", "th", "th_e", "buoyancy", "HMC"]

# Dictionary to store all variables
tracked_profiles = {}

# Populate all profiles in one loop
for type in types:
    for var in variables:
        tracked_profiles[f"{type}_profile_array_{var}"] = eval(f"{type}_profile_array_{var}")
        tracked_profiles[f"{type}_SE_{var}"] = eval(f"{type}_SE_{var}")
        tracked_profiles[f"nonCL_{type}_profile_array_{var}"] = eval(f"nonCL_{type}_profile_array_{var}")
        tracked_profiles[f"nonCL_{type}_SE_{var}"] = eval(f"nonCL_{type}_SE_{var}")

# Save all variables in an HDF5 file
dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out2/'
output_file=dir2+f"CL_nonCL_tracked_profiles_{res}_{Np_str}_5min_{job_id}.h5"
with h5py.File(output_file, "w") as h5f:
    for name, profile_data in tracked_profiles.items():
        h5f.create_dataset(name, data=profile_data)
print('done')


done


In [ ]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [135]:
if recombine==True:
    types = ["ALL", "SHALLOW", "DEEP"]
    variables = ["w", "qv", "qcqi", "th", "th_e", "buoyancy", "HMC"]
    
    vars_list = []
    SE_list = []
    
    for t in types:
        for var in variables:
            vars_list.append(f"{t}_profile_array_{var}")
            vars_list.append(f"nonCL_{t}_profile_array_{var}")
    for t in types:
        for var in variables:
            SE_list.append(f"{t}_SE_{var}")
            SE_list.append(f"nonCL_{t}_SE_{var}")

In [136]:
if recombine==True:
    dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out2/'
    
    
    #MAKING OUTPUT FILE PATH
    output_file=dir2+f"CL_nonCL_tracked_profiles_{res}_{Np_str}_5min.h5"
    
    #MAKING PROFILES DICTIONARY
    zhs = data['zh'].values
    profiles = {}  # Store profiles for all variables
    for var in vars_list:
        profiles[var] = np.zeros((len(zhs), 3))  # column 1: var, column 2: counter, column 3: list of zhs
        profiles[var][:, 2] = zhs 
    for var in SE_list:
        profiles[var] = np.zeros((len(zhs), 3))  # column 1: var, column 2: counter, column 3: list of zhs
        profiles[var][:, 2] = zhs 
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
    
        #CALLING IN DATA
        input_file=dir2+f"CL_nonCL_tracked_profiles_{res}_{Np_str}_5min_{job_id}.h5"
    
        #COMPILING PROFILES
        with h5py.File(input_file, 'r') as f:
            for var in vars_list + SE_list:  
                profiles[var][:,0:1+1]+=f[f'{var}'][:,0:1+1]
    
    #SAVING INTO FINAL FORM
    with h5py.File(output_file, 'w') as f:
        for var in profiles:
            profile_var = profiles[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

job_id = 10
job_id = 20
job_id = 30
job_id = 40
job_id = 50
job_id = 60


In [ ]:
#SBZ vs nonSBZ
################################################################################

In [86]:
#PLOTTING SBZ vs non-SBZ Vertical Profiles
##########################################

#SBZ
def SBZ_tracked_profile(var_data,type):

    if type=='all':
        out_arr=ALL_SBZ_out_arr.copy()
        # after_array=ALL_SBZ_out_after_array
    elif type=='shallow':
        out_arr=SHALLOW_SBZ_out_arr.copy()
        # after_array=SHALLOW_SBZ_out_after_array
    elif type=='deep':
        out_arr=DEEP_SBZ_out_arr.copy()
        # after_array=DEEP_SBZ_out_after_array
    
    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;
    
    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust]
        ys=Y[ts,p-index_adjust]
        xs=X[ts,p-index_adjust]
        vars=var_data[ts,p-index_adjust]
        np.add.at(profile_array[:, 0], zs, vars)
        np.add.at(profile_array[:, 1], zs, 1)
    return profile_array


print('ALL')
ALL_profile_array_w=SBZ_tracked_profile(W,type='all')
ALL_profile_array_qv=SBZ_tracked_profile(QV,type='all')
ALL_profile_array_qcqi=SBZ_tracked_profile(QCQI,type='all')
ALL_profile_array_th=SBZ_tracked_profile(TH,type='all')
ALL_profile_array_th_e=SBZ_tracked_profile(TH_E,type='all')
ALL_profile_array_buoyancy=SBZ_tracked_profile(BUOYANCY,type='all')
ALL_profile_array_HMC=SBZ_tracked_profile(HMC,type='all')

print('SHALLOW')
SHALLOW_profile_array_w=SBZ_tracked_profile(W,type='shallow')
SHALLOW_profile_array_qv=SBZ_tracked_profile(QV,type='shallow')
SHALLOW_profile_array_qcqi=SBZ_tracked_profile(QCQI,type='shallow')
SHALLOW_profile_array_th=SBZ_tracked_profile(TH,type='shallow')
SHALLOW_profile_array_th_e=SBZ_tracked_profile(TH_E,type='shallow')
SHALLOW_profile_array_buoyancy=SBZ_tracked_profile(BUOYANCY,type='shallow')
SHALLOW_profile_array_HMC=SBZ_tracked_profile(HMC,type='shallow')

print('DEEP')
DEEP_profile_array_w=SBZ_tracked_profile(W,type='deep')
DEEP_profile_array_qv=SBZ_tracked_profile(QV,type='deep')
DEEP_profile_array_qcqi=SBZ_tracked_profile(QCQI,type='deep')
DEEP_profile_array_th=SBZ_tracked_profile(TH,type='deep')
DEEP_profile_array_th_e=SBZ_tracked_profile(TH_E,type='deep')
DEEP_profile_array_buoyancy=SBZ_tracked_profile(BUOYANCY,type='deep')
DEEP_profile_array_HMC=SBZ_tracked_profile(HMC,type='deep')

ALL
SHALLOW
DEEP


In [87]:
POST_JOB_ARRAY=False
# POST_JOB_ARRAY=True

if POST_JOB_ARRAY==True:
    
    types = ["ALL", "SHALLOW", "DEEP"]
    variables = ["w", "qv", "qcqi", "th", "th_e", "buoyancy", "HMC"]
    
    vars_list = []
    SE_list = []
    
    for t in types:
        for var in variables:
            vars_list.append(f"{t}_profile_array_{var}")
            vars_list.append(f"nonSBZ_{t}_profile_array_{var}")
    for t in types:
        for var in variables:
            SE_list.append(f"{t}_SE_{var}")
            SE_list.append(f"nonSBZ_{t}_SE_{var}")
    
    # Define directory and output file path
    dir2 = dir + 'Project_Algorithms/Tracked_Profiles/job_out2/'
    output_file = dir2 + f"SBZ_nonSBZ_tracked_profiles_{res}_{Np_str}.h5"
    
    # Open the HDF5 file and read the stored datasets into dynamically named variables
    with h5py.File(output_file, 'r') as f:
        for var in vars_list:# + SE_list:
            globals()[var] = f[f'{var}'][:]

def averaged_profile_SE(profile): 

    ######################################## (MOVED TO SE_averaged_profilesssFUNCTION)
    where_undefined=np.where(profile[:,1]==1)
    mask=np.where(profile[:,1]!=0)
    #DIVIDE BY N = n-1
    with np.errstate(divide='ignore', invalid='ignore'):
        profile[mask, 0] /= (profile[mask, 1]-1) 
    #TAKE THE SQUARE ROOT
    profile[mask,0]=np.sqrt(profile[mask,0]) 
    #DIVIDE BY SQUARE ROOT OF n ==> STANDARD ERROR
    profile[mask, 0] /= np.sqrt(profile[mask, 1]) 
    profile[where_undefined,0]=0
    ########################################
    out_var=profile[ (profile[:, 1] != 0)]; #gets rid of rows that have no data
    SE=out_var[:, 0]
    zlevels=out_var[:, 2]
    return SE,zlevels

def SBZ_tracked_profile_SE(profile_data,var_data,type):    
    if type=='all':
        out_arr=ALL_SBZ_out_arr.copy()
        # after_array=ALL_SBZ_out_after_array
    elif type=='shallow':
        out_arr=SHALLOW_SBZ_out_arr.copy()
        # after_array=SHALLOW_SBZ_out_after_array
    elif type=='deep':
        out_arr=DEEP_SBZ_out_arr.copy()
        # after_array=DEEP_SBZ_out_after_array

    
    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;

    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust]
        ys=Y[ts,p-index_adjust]
        xs=X[ts,p-index_adjust]
        
        vars=var_data[ts,p-index_adjust]
        mean_mu=profile_data[zs,0]/profile_data[zs,1]
        np.add.at(profile_array[:, 0], zs,  (vars-mean_mu)**2) #SUMMING UP THE SQUARES
        np.add.at(profile_array[:, 1], zs,  1) #SUMMING UP THE SQUARES
    return profile_array


ALL_SE_w=SBZ_tracked_profile_SE(ALL_profile_array_w,W,type='all')
SHALLOW_SE_w=SBZ_tracked_profile_SE(SHALLOW_profile_array_w,W,type='shallow')
DEEP_SE_w=SBZ_tracked_profile_SE(DEEP_profile_array_w,W,type='deep')

ALL_SE_qv=SBZ_tracked_profile_SE(ALL_profile_array_qv,QV,type='all')
SHALLOW_SE_qv=SBZ_tracked_profile_SE(SHALLOW_profile_array_qv,QV,type='shallow')
DEEP_SE_qv=SBZ_tracked_profile_SE(DEEP_profile_array_qv,QV,type='deep')

ALL_SE_qcqi=SBZ_tracked_profile_SE(ALL_profile_array_qcqi,QCQI,type='all')
SHALLOW_SE_qcqi=SBZ_tracked_profile_SE(SHALLOW_profile_array_qcqi,QCQI,type='shallow')
DEEP_SE_qcqi=SBZ_tracked_profile_SE(DEEP_profile_array_qcqi,QCQI,type='deep')

ALL_SE_th=SBZ_tracked_profile_SE(ALL_profile_array_th,TH,type='all')
SHALLOW_SE_th=SBZ_tracked_profile_SE(SHALLOW_profile_array_th,TH,type='shallow')
DEEP_SE_th=SBZ_tracked_profile_SE(DEEP_profile_array_th,TH,type='deep')

ALL_SE_th_e=SBZ_tracked_profile_SE(ALL_profile_array_th_e,TH_E,type='all')
SHALLOW_SE_th_e=SBZ_tracked_profile_SE(SHALLOW_profile_array_th_e,TH_E,type='shallow')
DEEP_SE_th_e=SBZ_tracked_profile_SE(DEEP_profile_array_th_e,TH_E,type='deep')

ALL_SE_buoyancy=SBZ_tracked_profile_SE(ALL_profile_array_buoyancy,BUOYANCY,type='all')
SHALLOW_SE_buoyancy=SBZ_tracked_profile_SE(SHALLOW_profile_array_buoyancy,BUOYANCY,type='shallow')
DEEP_SE_buoyancy=SBZ_tracked_profile_SE(DEEP_profile_array_buoyancy,BUOYANCY,type='deep')

ALL_SE_HMC=SBZ_tracked_profile_SE(ALL_profile_array_HMC,HMC,type='all')
SHALLOW_SE_HMC=SBZ_tracked_profile_SE(SHALLOW_profile_array_HMC,HMC,type='shallow')
DEEP_SE_HMC=SBZ_tracked_profile_SE(DEEP_profile_array_HMC,HMC,type='deep')

In [88]:
#nonSBZ
def nonSBZ_tracked_profile(var_data,type):
    if type=='all':
        out_arr=ALL_nonSBZ_out_arr.copy()
        # after_array=ALL_nonSBZ_out_after_array
    elif type=='shallow':
        out_arr=SHALLOW_nonSBZ_out_arr.copy()
        # after_array=SHALLOW_nonSBZ_out_after_array
    elif type=='deep':
        out_arr=DEEP_nonSBZ_out_arr.copy()
        # after_array=DEEP_nonSBZ_out_after_array  

    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;
    
    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust]
        ys=Y[ts,p-index_adjust]
        xs=X[ts,p-index_adjust]
        vars=var_data[ts,p-index_adjust]
        np.add.at(profile_array[:, 0], zs, vars)
        np.add.at(profile_array[:, 1], zs, 1)
    return profile_array


print('ALL')
ALL_nonSBZ_profile_array_w=nonSBZ_tracked_profile(W,type='all')
ALL_nonSBZ_profile_array_qv=nonSBZ_tracked_profile(QV,type='all')
ALL_nonSBZ_profile_array_qcqi=nonSBZ_tracked_profile(QCQI,type='all')
ALL_nonSBZ_profile_array_th=nonSBZ_tracked_profile(TH,type='all')
ALL_nonSBZ_profile_array_th_e=nonSBZ_tracked_profile(TH_E,type='all')
ALL_nonSBZ_profile_array_buoyancy=nonSBZ_tracked_profile(BUOYANCY,type='all')
ALL_nonSBZ_profile_array_HMC=nonSBZ_tracked_profile(HMC,type='all')

print('SHALLOW')
SHALLOW_nonSBZ_profile_array_w=nonSBZ_tracked_profile(W,type='shallow')
SHALLOW_nonSBZ_profile_array_qv=nonSBZ_tracked_profile(QV,type='shallow')
SHALLOW_nonSBZ_profile_array_qcqi=nonSBZ_tracked_profile(QCQI,type='shallow')
SHALLOW_nonSBZ_profile_array_th=nonSBZ_tracked_profile(TH,type='shallow')
SHALLOW_nonSBZ_profile_array_th_e=nonSBZ_tracked_profile(TH_E,type='shallow')
SHALLOW_nonSBZ_profile_array_buoyancy=nonSBZ_tracked_profile(BUOYANCY,type='shallow')
SHALLOW_nonSBZ_profile_array_HMC=nonSBZ_tracked_profile(HMC,type='shallow')

print('DEEP')
DEEP_nonSBZ_profile_array_w=nonSBZ_tracked_profile(W,type='deep')
DEEP_nonSBZ_profile_array_qv=nonSBZ_tracked_profile(QV,type='deep')
DEEP_nonSBZ_profile_array_qcqi=nonSBZ_tracked_profile(QCQI,type='deep')
DEEP_nonSBZ_profile_array_th=nonSBZ_tracked_profile(TH,type='deep')
DEEP_nonSBZ_profile_array_th_e=nonSBZ_tracked_profile(TH_E,type='deep')
DEEP_nonSBZ_profile_array_buoyancy=nonSBZ_tracked_profile(BUOYANCY,type='deep')
DEEP_nonSBZ_profile_array_HMC=nonSBZ_tracked_profile(HMC,type='deep')

ALL
SHALLOW
DEEP


In [89]:
POST_JOB_ARRAY=False
# POST_JOB_ARRAY=True

if POST_JOB_ARRAY==True:
    
    types = ["ALL", "SHALLOW", "DEEP"]
    variables = ["w", "qv", "qcqi", "th", "th_e", "buoyancy", "HMC"]
    
    vars_list = []
    SE_list = []
    
    for t in types:
        for var in variables:
            vars_list.append(f"{t}_profile_array_{var}")
            vars_list.append(f"nonSBZ_{t}_profile_array_{var}")
    for t in types:
        for var in variables:
            SE_list.append(f"{t}_SE_{var}")
            SE_list.append(f"nonSBZ_{t}_SE_{var}")
    
    # Define directory and output file path
    dir2 = dir + 'Project_Algorithms/Tracked_Profiles/job_out2/'
    output_file = dir2 + f"SBZ_nonSBZ_tracked_profiles_{res}_{Np_str}.h5"
    
    # Open the HDF5 file and read the stored datasets into dynamically named variables
    with h5py.File(output_file, 'r') as f:
        for var in vars_list:# + SE_list:
            globals()[var] = f[f'{var}'][:]


def averaged_profile_SE(profile): 

    ######################################## (MOVED TO SE_averaged_profilesssFUNCTION)
    where_undefined=np.where(profile[:,1]==1)
    mask=np.where(profile[:,1]!=0)
    #DIVIDE BY N = n-1
    with np.errstate(divide='ignore', invalid='ignore'):
        profile[mask, 0] /= (profile[mask, 1]-1) 
    #TAKE THE SQUARE ROOT
    profile[mask,0]=np.sqrt(profile[mask,0]) 
    #DIVIDE BY SQUARE ROOT OF n ==> STANDARD ERROR
    profile[mask, 0] /= np.sqrt(profile[mask, 1]) 
    profile[where_undefined,0]=0
    ########################################
    out_var=profile[ (profile[:, 1] != 0)]; #gets rid of rows that have no data
    SE=out_var[:, 0]
    zlevels=out_var[:, 2]
    return SE,zlevels
    
def nonSBZ_tracked_profile_SE(profile_data,var_data,type):    
    if type=='all':
        out_arr=ALL_nonSBZ_out_arr.copy()
        # after_array=ALL_nonSBZ_out_after_array
    elif type=='shallow':
        out_arr=SHALLOW_nonSBZ_out_arr.copy()
        # after_array=SHALLOW_nonSBZ_out_after_array
    elif type=='deep':
        out_arr=DEEP_nonSBZ_out_arr.copy()
        # after_array=DEEP_nonSBZ_out_after_array

    
    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;

    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust]
        ys=Y[ts,p-index_adjust]
        xs=X[ts,p-index_adjust]
        
        vars=var_data[ts,p-index_adjust]
        mean_mu=profile_data[zs,0]/profile_data[zs,1]
        np.add.at(profile_array[:, 0], zs,  (vars-mean_mu)**2) #SUMMING UP THE SQUARES
        np.add.at(profile_array[:, 1], zs,  1) #SUMMING UP THE SQUARES
    return profile_array


nonSBZ_ALL_SE_w=nonSBZ_tracked_profile_SE(ALL_nonSBZ_profile_array_w,W,type='all')
nonSBZ_SHALLOW_SE_w=nonSBZ_tracked_profile_SE(SHALLOW_nonSBZ_profile_array_w,W,type='shallow')
nonSBZ_DEEP_SE_w=nonSBZ_tracked_profile_SE(DEEP_nonSBZ_profile_array_w,W,type='deep')

nonSBZ_ALL_SE_qv=nonSBZ_tracked_profile_SE(ALL_nonSBZ_profile_array_qv,QV,type='all')
nonSBZ_SHALLOW_SE_qv=nonSBZ_tracked_profile_SE(SHALLOW_nonSBZ_profile_array_qv,QV,type='shallow')
nonSBZ_DEEP_SE_qv=nonSBZ_tracked_profile_SE(DEEP_nonSBZ_profile_array_qv,QV,type='deep')

nonSBZ_ALL_SE_qcqi=nonSBZ_tracked_profile_SE(ALL_nonSBZ_profile_array_qcqi,QCQI,type='all')
nonSBZ_SHALLOW_SE_qcqi=nonSBZ_tracked_profile_SE(SHALLOW_nonSBZ_profile_array_qcqi,QCQI,type='shallow')
nonSBZ_DEEP_SE_qcqi=nonSBZ_tracked_profile_SE(DEEP_nonSBZ_profile_array_qcqi,QCQI,type='deep')

nonSBZ_ALL_SE_th=nonSBZ_tracked_profile_SE(ALL_nonSBZ_profile_array_th,TH,type='all')
nonSBZ_SHALLOW_SE_th=nonSBZ_tracked_profile_SE(SHALLOW_nonSBZ_profile_array_th,TH,type='shallow')
nonSBZ_DEEP_SE_th=nonSBZ_tracked_profile_SE(DEEP_nonSBZ_profile_array_th,TH,type='deep')

nonSBZ_ALL_SE_th_e=nonSBZ_tracked_profile_SE(ALL_nonSBZ_profile_array_th_e,TH_E,type='all')
nonSBZ_SHALLOW_SE_th_e=nonSBZ_tracked_profile_SE(SHALLOW_nonSBZ_profile_array_th_e,TH_E,type='shallow')
nonSBZ_DEEP_SE_th_e=nonSBZ_tracked_profile_SE(DEEP_nonSBZ_profile_array_th_e,TH_E,type='deep')

nonSBZ_ALL_SE_buoyancy=nonSBZ_tracked_profile_SE(ALL_nonSBZ_profile_array_buoyancy,BUOYANCY,type='all')
nonSBZ_SHALLOW_SE_buoyancy=nonSBZ_tracked_profile_SE(SHALLOW_nonSBZ_profile_array_buoyancy,BUOYANCY,type='shallow')
nonSBZ_DEEP_SE_buoyancy=nonSBZ_tracked_profile_SE(DEEP_nonSBZ_profile_array_buoyancy,BUOYANCY,type='deep')

nonSBZ_ALL_SE_HMC=nonSBZ_tracked_profile_SE(ALL_nonSBZ_profile_array_HMC,HMC,type='all')
nonSBZ_SHALLOW_SE_HMC=nonSBZ_tracked_profile_SE(SHALLOW_nonSBZ_profile_array_HMC,HMC,type='shallow')
nonSBZ_DEEP_SE_HMC=nonSBZ_tracked_profile_SE(DEEP_nonSBZ_profile_array_HMC,HMC,type='deep')

In [95]:
#SAVING

# Define categories and variables
types = ["ALL", "SHALLOW", "DEEP"]
variables = ["w", "qv", "qcqi", "th", "th_e", "buoyancy", "HMC"]

# Dictionary to store all variables
tracked_profiles = {}

# Populate all profiles in one loop
for type in types:
    for var in variables:
        tracked_profiles[f"{type}_profile_array_{var}"] = eval(f"{type}_profile_array_{var}")
        tracked_profiles[f"{type}_SE_{var}"] = eval(f"{type}_SE_{var}")
        tracked_profiles[f"nonSBZ_{type}_profile_array_{var}"] = eval(f"nonSBZ_{type}_profile_array_{var}")
        tracked_profiles[f"nonSBZ_{type}_SE_{var}"] = eval(f"nonSBZ_{type}_SE_{var}")

# Save all variables in an HDF5 file
dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out2/'
output_file=dir2+f"SBZ_nonSBZ_tracked_profiles_{res}_{Np_str}_5min_{job_id}.h5"
with h5py.File(output_file, "w") as h5f:
    for name, profile_data in tracked_profiles.items():
        h5f.create_dataset(name, data=profile_data)
print('done')


done


In [ ]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [96]:
if recombine==True:
    types = ["ALL", "SHALLOW", "DEEP"]
    variables = ["w", "qv", "qcqi", "th", "th_e", "buoyancy", "HMC"]
    
    vars_list = []
    SE_list = []
    
    for t in types:
        for var in variables:
            vars_list.append(f"{t}_profile_array_{var}")
            vars_list.append(f"nonSBZ_{t}_profile_array_{var}")
    for t in types:
        for var in variables:
            SE_list.append(f"{t}_SE_{var}")
            SE_list.append(f"nonSBZ_{t}_SE_{var}")

In [97]:
if recombine==True:
    dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out2/'
    
    
    #MAKING OUTPUT FILE PATH
    output_file=dir2+f"SBZ_nonSBZ_tracked_profiles_{res}_{Np_str}_5min.h5"
    
    #MAKING PROFILES DICTIONARY
    zhs = data['zh'].values
    profiles = {}  # Store profiles for all variables
    for var in vars_list:
        profiles[var] = np.zeros((len(zhs), 3))  # column 1: var, column 2: counter, column 3: list of zhs
        profiles[var][:, 2] = zhs 
    for var in SE_list:
        profiles[var] = np.zeros((len(zhs), 3))  # column 1: var, column 2: counter, column 3: list of zhs
        profiles[var][:, 2] = zhs 
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
    
        #CALLING IN DATA
        input_file=dir2+f"SBZ_nonSBZ_tracked_profiles_{res}_{Np_str}_5min_{job_id}.h5"
    
        #COMPILING PROFILES
        with h5py.File(input_file, 'r') as f:
            for var in vars_list + SE_list:  
                profiles[var][:,0:1+1]+=f[f'{var}'][:,0:1+1]
    
    #SAVING INTO FINAL FORM
    with h5py.File(output_file, 'w') as f:
        for var in profiles:
            profile_var = profiles[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")

In [ ]:
#ColdPool
################################################################

In [72]:
#PLOTTING SBZ vs non-SBZ Vertical Profiles
##########################################

#SBZ
def ColdPool_tracked_profile(var_data,type):

    if type=='all':
        out_arr=ALL_ColdPool_out_arr.copy()
        # after_array=ALL_ColdPool_after_array
    elif type=='shallow':
        out_arr=SHALLOW_ColdPool_out_arr
        # after_array=SHALLOW_ColdPool_after_array
    elif type=='deep':
        out_arr=DEEP_ColdPool_out_arr
        # after_array=DEEP_ColdPool_after_array
    
    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;
    
    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust]
        ys=Y[ts,p-index_adjust]
        xs=X[ts,p-index_adjust]
        vars=var_data[ts,p-index_adjust]
        np.add.at(profile_array[:, 0], zs, vars)
        np.add.at(profile_array[:, 1], zs, 1)
    return profile_array

print('ALL')
ALL_ColdPool_profile_array_w = ColdPool_tracked_profile(W, type='all')
ALL_ColdPool_profile_array_qv = ColdPool_tracked_profile(QV, type='all')
ALL_ColdPool_profile_array_qcqi = ColdPool_tracked_profile(QCQI, type='all')
ALL_ColdPool_profile_array_th = ColdPool_tracked_profile(TH, type='all')
ALL_ColdPool_profile_array_th_e = ColdPool_tracked_profile(TH_E, type='all')
ALL_ColdPool_profile_array_buoyancy = ColdPool_tracked_profile(BUOYANCY, type='all')
ALL_ColdPool_profile_array_HMC = ColdPool_tracked_profile(HMC, type='all')

print('DEEP')
DEEP_ColdPool_profile_array_w = ColdPool_tracked_profile(W, type='deep')
DEEP_ColdPool_profile_array_qv = ColdPool_tracked_profile(QV, type='deep')
DEEP_ColdPool_profile_array_qcqi = ColdPool_tracked_profile(QCQI, type='deep')
DEEP_ColdPool_profile_array_th = ColdPool_tracked_profile(TH, type='deep')
DEEP_ColdPool_profile_array_th_e = ColdPool_tracked_profile(TH_E, type='deep')
DEEP_ColdPool_profile_array_buoyancy = ColdPool_tracked_profile(BUOYANCY, type='deep')
DEEP_ColdPool_profile_array_HMC = ColdPool_tracked_profile(HMC, type='deep')

print('SHALLOW')
SHALLOW_ColdPool_profile_array_w = ColdPool_tracked_profile(W, type='shallow')
SHALLOW_ColdPool_profile_array_qv = ColdPool_tracked_profile(QV, type='shallow')
SHALLOW_ColdPool_profile_array_qcqi = ColdPool_tracked_profile(QCQI, type='shallow')
SHALLOW_ColdPool_profile_array_th = ColdPool_tracked_profile(TH, type='shallow')
SHALLOW_ColdPool_profile_array_th_e = ColdPool_tracked_profile(TH_E, type='shallow')
SHALLOW_ColdPool_profile_array_buoyancy = ColdPool_tracked_profile(BUOYANCY, type='shallow')
SHALLOW_ColdPool_profile_array_HMC = ColdPool_tracked_profile(HMC, type='shallow')

ALL
DEEP
SHALLOW


In [73]:
POST_JOB_ARRAY=False
# POST_JOB_ARRAY=True

if POST_JOB_ARRAY==True:
        
    types = ["ALL", "SHALLOW", "DEEP"]
    variables = ["w", "qv", "qcqi", "th", "th_e", "buoyancy", "HMC"]
    
    vars_list = []
    SE_list = []
    
    for t in types:
        for var in variables:
            vars_list.append(f"{t}_profile_array_{var}")
            vars_list.append(f"nonSBZ_{t}_profile_array_{var}")
    for t in types:
        for var in variables:
            SE_list.append(f"{t}_SE_{var}")
            SE_list.append(f"nonSBZ_{t}_SE_{var}")
    
    # Define directory and output file path
    dir2 = dir + 'Project_Algorithms/Tracked_Profiles/job_out2/'
    output_file = dir2 + f"SBZ_nonSBZ_tracked_profiles_{res}_{Np_str}.h5"
    
    # Open the HDF5 file and read the stored datasets into dynamically named variables
    with h5py.File(output_file, 'r') as f:
        for var in vars_list + SE_list:
            globals()[var] = f[f'{var}'][:]
    


def averaged_profile_SE(profile): 

    ######################################## (MOVED TO SE_averaged_profilesssFUNCTION)
    where_undefined=np.where(profile[:,1]==1)
    mask=np.where(profile[:,1]!=0)
    #DIVIDE BY N = n-1
    with np.errstate(divide='ignore', invalid='ignore'):
        profile[mask, 0] /= (profile[mask, 1]-1) 
    #TAKE THE SQUARE ROOT
    profile[mask,0]=np.sqrt(profile[mask,0]) 
    #DIVIDE BY SQUARE ROOT OF n ==> STANDARD ERROR
    profile[mask, 0] /= np.sqrt(profile[mask, 1]) 
    profile[where_undefined,0]=0
    ########################################
    out_var=profile[ (profile[:, 1] != 0)]; #gets rid of rows that have no data
    SE=out_var[:, 0]
    zlevels=out_var[:, 2]
    return SE,zlevels
    
def ColdPool_tracked_profile_SE(profile_data,var_data,type):    

    if type=='all':
        out_arr=ALL_ColdPool_out_arr.copy()
        # after_array=ALL_ColdPool_after_array
    elif type=='shallow':
        out_arr=SHALLOW_ColdPool_out_arr
        # after_array=SHALLOW_ColdPool_after_array
    elif type=='deep':
        out_arr=DEEP_ColdPool_out_arr
        # after_array=DEEP_ColdPool_after_array
    
    zhs=data['zh'].values
    profile_array =np.zeros((len(zhs), 3)) #column 1: var, column 2: counter, column 3: list of zhs
    profile_array[:,2]=zhs;

    for row in range(out_arr.shape[0]):
        after=out_arr[row,3]
        # if np.mod(row,3000)==0: print(f'{row}/{out_arr.shape[0]}')
        p=out_arr[row,0]
        
        # ts=np.arange(out_arr[row,4],out_arr[row,5]+1 + after)
        ts_end = min(out_arr[row, 2] + 1 + after, len(data['time'])) #this takes care of exceeding buffers
        ts = np.arange(out_arr[row, 1], ts_end)
        
        zs=Z[ts,p-index_adjust]
        ys=Y[ts,p-index_adjust]
        xs=X[ts,p-index_adjust]
        
        vars=var_data[ts,p-index_adjust]
        mean_mu=profile_data[zs,0]/profile_data[zs,1]
        np.add.at(profile_array[:, 0], zs,  (vars-mean_mu)**2) #SUMMING UP THE SQUARES
        np.add.at(profile_array[:, 1], zs,  1) #SUMMING UP THE SQUARES

    return profile_array


ALL_ColdPool_SE_w = ColdPool_tracked_profile_SE(ALL_ColdPool_profile_array_w, W, type='all')
SHALLOW_ColdPool_SE_w = ColdPool_tracked_profile_SE(SHALLOW_ColdPool_profile_array_w, W, type='shallow')
DEEP_ColdPool_SE_w = ColdPool_tracked_profile_SE(DEEP_ColdPool_profile_array_w, W, type='deep')

ALL_ColdPool_SE_qv = ColdPool_tracked_profile_SE(ALL_ColdPool_profile_array_qv, QV, type='all')
SHALLOW_ColdPool_SE_qv = ColdPool_tracked_profile_SE(SHALLOW_ColdPool_profile_array_qv, QV, type='shallow')
DEEP_ColdPool_SE_qv = ColdPool_tracked_profile_SE(DEEP_ColdPool_profile_array_qv, QV, type='deep')

ALL_ColdPool_SE_qcqi = ColdPool_tracked_profile_SE(ALL_ColdPool_profile_array_qcqi, QCQI, type='all')
SHALLOW_ColdPool_SE_qcqi = ColdPool_tracked_profile_SE(SHALLOW_ColdPool_profile_array_qcqi, QCQI, type='shallow')
DEEP_ColdPool_SE_qcqi = ColdPool_tracked_profile_SE(DEEP_ColdPool_profile_array_qcqi, QCQI, type='deep')

ALL_ColdPool_SE_th = ColdPool_tracked_profile_SE(ALL_ColdPool_profile_array_th, TH, type='all')
SHALLOW_ColdPool_SE_th = ColdPool_tracked_profile_SE(SHALLOW_ColdPool_profile_array_th, TH, type='shallow')
DEEP_ColdPool_SE_th = ColdPool_tracked_profile_SE(DEEP_ColdPool_profile_array_th, TH, type='deep')

ALL_ColdPool_SE_th_e = ColdPool_tracked_profile_SE(ALL_ColdPool_profile_array_th_e, TH_E, type='all')
SHALLOW_ColdPool_SE_th_e = ColdPool_tracked_profile_SE(SHALLOW_ColdPool_profile_array_th_e, TH_E, type='shallow')
DEEP_ColdPool_SE_th_e = ColdPool_tracked_profile_SE(DEEP_ColdPool_profile_array_th_e, TH_E, type='deep')

ALL_ColdPool_SE_buoyancy = ColdPool_tracked_profile_SE(ALL_ColdPool_profile_array_buoyancy, BUOYANCY, type='all')
SHALLOW_ColdPool_SE_buoyancy = ColdPool_tracked_profile_SE(SHALLOW_ColdPool_profile_array_buoyancy, BUOYANCY, type='shallow')
DEEP_ColdPool_SE_buoyancy = ColdPool_tracked_profile_SE(DEEP_ColdPool_profile_array_buoyancy, BUOYANCY, type='deep')

ALL_ColdPool_SE_HMC = ColdPool_tracked_profile_SE(ALL_ColdPool_profile_array_HMC, HMC, type='all')
SHALLOW_ColdPool_SE_HMC = ColdPool_tracked_profile_SE(SHALLOW_ColdPool_profile_array_HMC, HMC, type='shallow')
DEEP_ColdPool_SE_HMC = ColdPool_tracked_profile_SE(DEEP_ColdPool_profile_array_HMC, HMC, type='deep')


In [112]:
#SAVING

# Define categories and variables
types = ["ALL", "SHALLOW", "DEEP"]
variables = ["w", "qv", "qcqi", "th", "th_e", "buoyancy", "HMC"]

# Dictionary to store all variables
tracked_profiles = {}

# Populate all profiles in one loop
for type in types:
    for var in variables:
        tracked_profiles[f"{type}_ColdPool_profile_array_{var}"] = eval(f"{type}_ColdPool_profile_array_{var}")
        tracked_profiles[f"{type}_ColdPool_SE_{var}"] = eval(f"{type}_ColdPool_SE_{var}")

# Save all variables in an HDF5 file
dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out2/'
output_file=dir2+f"ColdPool_tracked_profiles_{res}_{Np_str}_5min_{job_id}.h5"
with h5py.File(output_file, "w") as h5f:
    for name, profile_data in tracked_profiles.items():
        h5f.create_dataset(name, data=profile_data)
print('done')


done


In [ ]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [113]:
recombine=True:
    types = ["ALL", "SHALLOW", "DEEP"]
    variables = ["w", "qv", "qcqi", "th", "th_e", "buoyancy", "HMC"]
    
    vars_list = []
    SE_list = []
    
    for t in types:
        for var in variables:
            vars_list.append(f"{t}_ColdPool_profile_array_{var}")
    for t in types:
        for var in variables:
            SE_list.append(f"{t}_ColdPool_SE_{var}")

In [116]:
recombine=True:
    dir2=dir+'Project_Algorithms/Tracked_Profiles/job_out2/'
    
    #MAKING OUTPUT FILE PATH
    output_file=dir2+f"ColdPool_tracked_profiles_{res}_{Np_str}_5min.h5"
    
    #MAKING PROFILES DICTIONARY
    zhs = data['zh'].values
    profiles = {}  # Store profiles for all variables
    for var in vars_list:
        profiles[var] = np.zeros((len(zhs), 3))  # column 1: var, column 2: counter, column 3: list of zhs
        profiles[var][:, 2] = zhs 
    for var in SE_list:
        profiles[var] = np.zeros((len(zhs), 3))  # column 1: var, column 2: counter, column 3: list of zhs
        profiles[var][:, 2] = zhs 
    
    num_jobs=60
    for job_id in np.arange(1,num_jobs+1):
        if np.mod(job_id,10)==0: print(f"job_id = {job_id}")
    
        #CALLING IN DATA
        input_file=dir2+f"ColdPool_tracked_profiles_{res}_{Np_str}_5min_{job_id}.h5"
    
        #COMPILING PROFILES
        with h5py.File(input_file, 'r') as f:
            for var in vars_list + SE_list:  
                profiles[var][:,0:1+1]+=f[f'{var}'][:,0:1+1]
    
    #SAVING INTO FINAL FORM
    with h5py.File(output_file, 'w') as f:
        for var in profiles:
            profile_var = profiles[var]
            f.create_dataset(f'{var}', data=profile_var, compression="gzip")